# 06b — M2 (statistical global): PTB-XL single-dataset detector

Clean rebuild of legacy 08b, English. A **full standalone single-dataset detector** trained on PTB-XL (folds 1-8); evaluated same-dataset (PTB→PTB, fold 9) and cross-dataset (PTB→Ningbo, folds 1-8) for the batch effect. Event-free representation. Features reloaded from the shared `m2_features.csv` (no re-extraction). Judge: AP. **PTB fold 10 untouched; Ningbo folds 9-10 untouched.**

> **Selection note — differs from 06a / 05a on purpose.** This is a standalone detector optimized for its OWN performance. On the large single-dataset M2 pool (1452 → ~450 after gate+dedup) with few WPW (57), the bootstrap confidence intervals are very wide, so the **1-SE parsimony rule** used by the combined/deployed detectors over-prunes and weakens the model. Here we therefore select K and config by **maximizing OOF AP under the overfit-gap guard** (gap ≤ smallest achievable + 0.10), as in legacy 08b — the strongest standalone model. The combined detector (06a, deployed in the ensemble) keeps the 1-SE rule.

**Outputs:** `models/M2_ptbxl/m2_ptbxl_*`, `reports/{figures,metrics}/M2_ptbxl_*`.

> Legacy 08b kept as reference. Narrative: see `JOURNAL.md`.

### Block 6b.0: Setup + load features (PTB-XL)

In [ ]:
# M2 (statistical global) - SINGLE-DATASET detector. Clean rebuild of legacy 08b/08c, English, with the
# unified parsimony selection (1-SE rule, identical to M1 05b/c and M2 06a). Trains on one dataset
# (folds 1-8), evaluates same-dataset (fold 9) AND cross-dataset transfer to the OTHER dataset
# (folds 1-8) to quantify the batch effect. Features reloaded from the shared m2_features.csv (no
# re-extraction). Judge: AP. Fold 10 (both datasets) + fold 9 of the cross dataset NEVER touched.
import os, sys, json
import numpy as np, pandas as pd
ROOT      = r'.'
PROCESSED = os.path.join(ROOT,'data','processed')
SRC       = os.path.join(ROOT,'src')
FIG       = os.path.join(ROOT,'reports','figures')
METRICS   = os.path.join(ROOT,'reports','metrics')
sys.path.insert(0, SRC)
from evaluation import evaluate_standard, f1max_threshold

TRAIN_SRC, CROSS_SRC = 'ptbxl', 'ningbo'    # 06b = PTB detector  (06c uses: 'ningbo', 'ptbxl')
TAG = TRAIN_SRC
FEATURES_CSV = os.path.join(PROCESSED,'m2_features.csv')
META = ['ecg_id','patient_id','label','fold','source','extraction_failed']
LEGACY = {'ptbxl': {'same_AP':0.367,'cross_AP':0.209},
          'ningbo':{'same_AP':0.428,'cross_AP':0.218}}[TRAIN_SRC]

dfall = pd.read_csv(FEATURES_CSV, dtype={'ecg_id':str})
df  = dfall[dfall.source==TRAIN_SRC].reset_index(drop=True)     # training data (same)
dfx = dfall[dfall.source==CROSS_SRC].reset_index(drop=True)     # cross-dataset data
ALL_FEATS = [c for c in dfall.columns if c not in META]
print(f"M2 single-dataset detector: train={TRAIN_SRC} | cross-test={CROSS_SRC}")
print(f"{TRAIN_SRC}: {len(df)} ECGs, {int((df.label==1).sum())} WPW | WPW/fold {df[df.label==1].groupby('fold').size().to_dict()}")
print(f"{CROSS_SRC} (cross): {len(dfx)} ECGs, {int((dfx.label==1).sum())} WPW")
print(f"Available features: {len(ALL_FEATS)}")


### Block 6b.0b: Patient-leakage check (blocking assert)

In [ ]:
# Patient-leakage check within the training dataset (blocking assert).
pf = df.groupby('patient_id')['fold'].nunique(); leaky = pf[pf>1]
print(f"{TRAIN_SRC}: {df.patient_id.nunique()} patients, {len(df)} ECGs | in >1 fold: {len(leaky)}")
assert len(leaky)==0, f"PATIENT LEAK {TRAIN_SRC}: {len(leaky)} patients in multiple folds."
print("[OK] No patient leakage.")


### Block 6b.2: Feature selection (single-dataset FDR gate + dedup)

In [ ]:
# Feature selection on TRAIN_SRC folds 1-8: SINGLE-DATASET gate |Cohen's d|>0.3 AND p_FDR<0.05 (BH) AND
# IC95 bootstrap of d excludes 0 (NO cross-dataset criterion - single source). Fast Spearman>0.9 dedup.
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
SEL_CSV = os.path.join(PROCESSED, f'm2_{TAG}_selection.csv')   # reused from legacy (identical logic)
tr = df[df.fold.between(1,8)]
print(f"Train folds 1-8 ({TRAIN_SRC}): {len(tr)} ECGs, {int((tr.label==1).sum())} WPW")

def cohens_d(a,b):
    a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return np.nan
    sp=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    return (a.mean()-b.mean())/sp if sp>0 else np.nan
def d_ci(a,b,n=1000,seed=42):
    rng=np.random.default_rng(seed); a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return (np.nan,np.nan)
    ds=[cohens_d(rng.choice(a,len(a),True),rng.choice(b,len(b),True)) for _ in range(n)]
    return tuple(np.nanpercentile(ds,[2.5,97.5]))

if os.path.exists(SEL_CSV):
    res=pd.read_csv(SEL_CSV); print(f"[guard] {SEL_CSV} reloaded ({len(res)} features).")
else:
    w,nn=tr[tr.label==1],tr[tr.label==0]; rows=[]
    for c in tqdm(ALL_FEATS, desc='Gate (d+p+IC)', unit='feat'):
        a,b=w[c].values.astype(float),nn[c].values.astype(float)
        d=cohens_d(a,b)
        try: _,p=mannwhitneyu(a[~np.isnan(a)],b[~np.isnan(b)],alternative='two-sided')
        except Exception: p=np.nan
        lo,hi=d_ci(a,b); ci_ok=(np.isfinite(lo) and np.isfinite(hi) and (lo>0)==(hi>0))
        rows.append({'feature':c,'d':d,'p_raw':p,'ci_excl0':ci_ok})
    res=pd.DataFrame(rows); ok=res.p_raw.notna()
    res.loc[ok,'p_FDR']=multipletests(res.loc[ok,'p_raw'],method='fdr_bh')[1]
    res['gate']=(res.d.abs()>0.3)&(res.p_FDR<0.05)&res.ci_excl0
    res=res.sort_values('d',key=lambda s:s.abs(),ascending=False).reset_index(drop=True)
    res.to_csv(SEL_CSV,index=False)
passed=res[res.gate].feature.tolist()
print(f"Pass the gate: {len(passed)}")

def dedup_fast(feats):
    sub=tr[feats].rank().to_numpy(); cm=np.nanmean(sub,axis=0)
    ii=np.where(np.isnan(sub)); sub[ii]=np.take(cm,ii[1])
    C=np.abs(np.corrcoef(sub,rowvar=False)); idx={f:i for i,f in enumerate(feats)}; keep=[]
    for f in feats:
        if all(C[idx[f],idx[k]]<=0.9 for k in keep): keep.append(f)
    return keep
dedup_list = dedup_fast(passed) if passed else []
print(f"After dedup>0.9: {len(dedup_list)} features (= k_max)\n")
pd.set_option('display.float_format',lambda x:f'{x:.3f}')
print(res[['feature','d','p_FDR','gate']].head(15).to_string(index=False))


### Block 6b.3: AP-vs-k (same) + cross-dataset batch-effect curve

In [ ]:
# AP-vs-k: AP OOF same-dataset (provisional K, 95% plateau) + AP/AUC cross-dataset (batch-effect viz:
# the AP collapses while the AUC holds). Guard: reload the k-curve CSV. The provisional K seeds 6x.3d.
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier
from tqdm import tqdm
import matplotlib.pyplot as plt
RK_CSV = os.path.join(PROCESSED, f'm2_{TAG}_ap_vs_k.csv')   # reused from legacy (identical logic)

d8 = df[df.fold.between(1,8)].reset_index(drop=True)
y, folds = d8.label.values, d8.fold.values
FOLD_MASKS=[(folds!=h,folds==h) for h in sorted(np.unique(folds))]
spw8=(y==0).sum()/max((y==1).sum(),1)
dcx = dfx[dfx.fold.between(1,8)].reset_index(drop=True); ycx=dcx.label.values   # cross folds 1-8 (cross 9-10 untouched)

def make_xgb(spw=None,**kw):
    if spw is None: spw=spw8
    p=dict(n_estimators=100,max_depth=2,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,
           reg_lambda=2.0,min_child_weight=3,scale_pos_weight=spw,eval_metric='aucpr',
           tree_method='hist',random_state=42,n_jobs=10); p.update(kw); return XGBClassifier(**p)
def oof_same(feats,**kw):
    X=d8[feats]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        spw=(y[trm]==0).sum()/max((y[trm]==1).sum(),1)
        oof[vam]=make_xgb(spw,**kw).fit(X[trm],y[trm]).predict_proba(X[vam])[:,1]
    return oof

k_max=len(dedup_list)
if os.path.exists(RK_CSV):
    rk=pd.read_csv(RK_CSV); print(f"[guard] {RK_CSV} reloaded ({len(rk)} k).")
else:
    rows=[]
    for k in tqdm(range(1,k_max+1), desc='k (same+cross)', unit='k'):
        feats=dedup_list[:k]
        oof=oof_same(feats); msk=~np.isnan(oof)
        ap_same=average_precision_score(y[msk],oof[msk]); auc_same=roc_auc_score(y[msk],oof[msk])
        mdl=make_xgb().fit(d8[feats],y); pc=mdl.predict_proba(dcx[feats])[:,1]
        ap_cross=average_precision_score(ycx,pc); auc_cross=roc_auc_score(ycx,pc)
        rows.append({'k':k,'AP_same':ap_same,'AUC_same':auc_same,'AP_cross':ap_cross,'AUC_cross':auc_cross})
        if k%20==0: pd.DataFrame(rows).to_csv(RK_CSV,index=False)
    rk=pd.DataFrame(rows); rk.to_csv(RK_CSV,index=False)

# Triple curve (the batch effect across all k)
fig,ax=plt.subplots(figsize=(13,6))
ax.plot(rk.k,rk.AP_same,'o-',color='#2563eb',ms=3,label=f'AP OOF same ({TRAIN_SRC}) - chooses k')
ax.plot(rk.k,rk.AP_cross,'s-',color='#dc2626',ms=3,label=f'AP cross ({CROSS_SRC}) - collapses')
ax.plot(rk.k,rk.AUC_cross,'^--',color='#16a34a',ms=3,label=f'AUC cross ({CROSS_SRC}) - holds (detection transfers)')
ax.set(xlabel='k',ylabel='metric'); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f'M2 {TRAIN_SRC} - AP same vs AP/AUC cross ({CROSS_SRC})')
plt.savefig(os.path.join(FIG,f'm2_{TAG}_ap_vs_k.png'),dpi=150,bbox_inches='tight'); plt.show()

# provisional K on AP OOF same (95% plateau) -> seeds the 6x.3d grid
kbest=rk.loc[rk.AP_same.idxmax()]; K=int(rk[rk.AP_same>=0.95*kbest.AP_same].k.min())
print(f"\nMax AP same at k={int(kbest.k)} (AP={kbest.AP_same:.3f}). 95% plateau from k={K} (provisional).")
print(f"At k={K}: cross AP {rk[rk.k==K].AP_cross.values[0]:.3f} vs same AP {rk[rk.k==K].AP_same.values[0]:.3f} -> collapse = batch effect.")
FEATURES_K=dedup_list[:K]


### Block 6b.3d: 2D K x config grid -> best standalone model (max OOF under gap)

In [ ]:
# 2D K x config grid -> best standalone same-dataset model. diag_one returns OOF AP / train AP / gap /
# fold9 AP. Selection = MAXIMIZE OOF AP under the overfit-gap guard (gap <= smallest achievable + 0.10).
# NB: this DIFFERS from the combined/deployed detectors (06a, 05a), which use the 1-SE parsimony rule.
# On the large single-dataset M2 pool (~450 features) with few WPW (57-58), the bootstrap IC is too wide
# for 1-SE (it over-prunes and weakens the model); here we keep the STRONGEST standalone model. See the
# note at the top of the notebook. Reuses the legacy grid cache (5 configs) -> reproduces legacy numbers.
from tqdm import tqdm
G2_CSV = os.path.join(PROCESSED, f'm2_{TAG}_grid2d.csv')
y8 = d8.label.values
f9 = df[df.fold==9].reset_index(drop=True); y9 = f9.label.values

def diag_one(cfg, feats):
    X8=d8[feats]; X9=f9[feats]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y8[trm].sum()==0 or y8[vam].sum()==0: continue
        spw=(y8[trm]==0).sum()/max((y8[trm]==1).sum(),1)
        oof[vam]=make_xgb(spw,**cfg).fit(X8[trm],y8[trm]).predict_proba(X8[vam])[:,1]
    m=~np.isnan(oof)
    ap_oof=average_precision_score(y8[m],oof[m])
    mdl=make_xgb(**cfg).fit(X8,y8)
    ap_tr=average_precision_score(y8,mdl.predict_proba(X8)[:,1])
    ap_f9=average_precision_score(y9,mdl.predict_proba(X9)[:,1]) if y9.sum()>0 else np.nan
    return ap_oof,ap_tr,ap_tr-ap_oof,ap_f9

CFGS={'d2_lr02_ne300':dict(max_depth=2,learning_rate=0.02,n_estimators=300,min_child_weight=3),
      'd2_lr03_ne200':dict(max_depth=2,learning_rate=0.03,n_estimators=200,min_child_weight=3),
      'd2_lr03_ne300':dict(max_depth=2,learning_rate=0.03,n_estimators=300,min_child_weight=3),
      'd2_lr05_ne200':dict(max_depth=2,learning_rate=0.05,n_estimators=200,min_child_weight=3),
      'd3_lr03_ne200':dict(max_depth=3,learning_rate=0.03,n_estimators=200,min_child_weight=3)}

if os.path.exists(G2_CSV):
    G2=pd.read_csv(G2_CSV); print(f"[guard] {G2_CSV} reloaded ({len(G2)} combos).")
else:
    Kgrid=sorted(set([max(5,K//2),K,min(k_max,int(K*1.5)),k_max]))
    rows=[]
    for kk in tqdm(Kgrid, desc='K x config', unit='K'):
        for cn,c in CFGS.items():
            ao,at,gp,af=diag_one(c,dedup_list[:kk])
            rows.append({'K':kk,'cfg':cn,'AP_oof':ao,'AP_train':at,'gap':gp,'AP_f9':af})
    G2=pd.DataFrame(rows); G2.to_csv(G2_CSV,index=False)
pd.set_option('display.float_format',lambda x:f'{x:.4f}')
print(G2.sort_values('AP_oof',ascending=False).head(10).to_string(index=False))

# Selection: max OOF AP under the gap guard (gap <= smallest achievable + 0.10). Best standalone model.
gap_floor=float(G2.gap.min()); gap_cap=gap_floor+0.10
ok=G2[G2.gap<=gap_cap]
best=ok.sort_values('AP_oof',ascending=False).iloc[0]
K=int(best.K); FEATURES_K=dedup_list[:K]
CFG={**CFGS[best.cfg],'subsample':0.8,'colsample_bytree':0.8,'reg_lambda':2.0}
print(f"\n[selection] best standalone model: max OOF AP under gap <= gap_floor+0.10 (gap_floor={gap_floor:.3f}, cap={gap_cap:.3f})")
print(f">>> FINAL {TRAIN_SRC}: K={K}, cfg={best.cfg} (gap {best.gap:.3f}, AP_oof {best.AP_oof:.3f}) - strongest standalone model")


### Block 6b.4: Same-dataset model + standardized evaluation (fold 9)

In [ ]:
# Final same-dataset model + standardized evaluation (fold 9). Raw model (folds 1-8) for threshold/cross
# test; OOF native folds for F1-max threshold + Platt calibration (anti-shuffle); multi-seed; evaluate_standard.
from sklearn.linear_model import LogisticRegression as _LR
X8,X9=d8[FEATURES_K],f9[FEATURES_K]
def make_final(**kw):
    p=dict(**CFG,scale_pos_weight=spw8,eval_metric='aucpr',tree_method='hist',
           random_state=42,n_jobs=10); p.update(kw); return XGBClassifier(**p)
xgb_raw=make_final().fit(X8,y8); score9=xgb_raw.predict_proba(X9)[:,1]
ap_train=average_precision_score(y8,xgb_raw.predict_proba(X8)[:,1])
oof_raw=np.full(len(d8),np.nan)
for h in sorted(np.unique(folds)):
    trm,vam=folds!=h,folds==h
    if y8[trm].sum()==0 or y8[vam].sum()==0: continue
    spw=(y8[trm]==0).sum()/max((y8[trm]==1).sum(),1)
    oof_raw[vam]=make_final(scale_pos_weight=spw).fit(X8[trm],y8[trm]).predict_proba(X8[vam])[:,1]
mask_oof=~np.isnan(oof_raw)
platt=_LR(max_iter=2000).fit(oof_raw[mask_oof].reshape(-1,1),y8[mask_oof])
score9_cal=platt.predict_proba(score9.reshape(-1,1))[:,1]
aps=[]; aucs=[]
for s in range(10):
    mm=make_final(random_state=s).fit(X8,y8); p=mm.predict_proba(X9)[:,1]
    aps.append(average_precision_score(y9,p)); aucs.append(roc_auc_score(y9,p))
MULTISEED=dict(AP_mean=float(np.mean(aps)),AP_std=float(np.std(aps)),AUC_mean=float(np.mean(aucs)),AUC_std=float(np.std(aucs)))
print(f"Multi-seed fold9 same: AP {np.mean(aps):.3f}+/-{np.std(aps):.3f} | AUC {np.mean(aucs):.3f}+/-{np.std(aucs):.3f}")
SAME_METRICS=evaluate_standard(
    name=f'M2_{TAG}',
    y_oof=y8[mask_oof], score_oof=oof_raw[mask_oof], y_test=y9, score_test=score9,
    figures_dir=FIG, metrics_dir=METRICS,
    score_test_calibrated=score9_cal, ap_train=ap_train, multiseed=MULTISEED,
    extra={'K':K,'xgb_params':{**CFG},'eval_type':'same_dataset_fold9','legacy_08bc_same_AP':LEGACY['same_AP']})


### Block 6b.4-cross: Cross-dataset transfer (batch effect)

In [ ]:
# Cross-dataset transfer: the TRAIN_SRC-trained model applied to CROSS_SRC folds 1-8 (cross 9-10 untouched).
# Expect AUC to hold (detection transfers) but AP to collapse (score scale does not transfer = batch effect).
score_cross = xgb_raw.predict_proba(dcx[FEATURES_K])[:,1]
score_cross_cal = platt.predict_proba(score_cross.reshape(-1,1))[:,1]
CROSS_METRICS=evaluate_standard(
    name=f'M2_{TAG}_CROSS_on_{CROSS_SRC}',
    y_oof=y8[mask_oof], score_oof=oof_raw[mask_oof],   # F1-max threshold on OOF same (consistent)
    y_test=ycx, score_test=score_cross,
    figures_dir=FIG, metrics_dir=METRICS, score_test_calibrated=score_cross_cal,
    extra={'eval_type':'cross_dataset','train_on':TRAIN_SRC,'test_on':f'{CROSS_SRC}_folds1_8'})
print("\n=== same vs cross (the batch-effect result) ===")
print(f"  SAME  ({TRAIN_SRC} fold9): AP {SAME_METRICS['AP']:.3f}  AUC {SAME_METRICS['AUC']:.3f}")
print(f"  CROSS ({CROSS_SRC} f1-8) : AP {CROSS_METRICS['AP']:.3f}  AUC {CROSS_METRICS['AUC']:.3f}")
print(f"  -> AP collapses ({SAME_METRICS['AP']:.3f} -> {CROSS_METRICS['AP']:.3f}) while AUC holds "
      f"({SAME_METRICS['AUC']:.3f} -> {CROSS_METRICS['AUC']:.3f}). Detection transfers, score scale does not.")
print("  -> justifies the COMBINED detector (06a) + the Flask percentile display.")


### Block 6b.5: Freeze

In [ ]:
# Freeze the single-dataset M2 detector (model + Platt + features + config with same & cross metrics).
# Overwrites the canonical M2_ptbxl / M2_ningbo artifacts. Fold 10 (both) + fold 9 cross never touched.
import joblib
from datetime import datetime
MDIR=os.path.join(ROOT,'models',f'M2_{TAG}'); os.makedirs(MDIR,exist_ok=True)
joblib.dump(xgb_raw,os.path.join(MDIR,f'm2_{TAG}_xgb_raw.joblib'))
joblib.dump(platt,os.path.join(MDIR,f'm2_{TAG}_platt.joblib'))
pd.Series(FEATURES_K,name='feature').to_csv(os.path.join(MDIR,f'm2_{TAG}_features.csv'),index=False)
config={
 "model":f"M2_{TAG}_pure",
 "train_on":f"{TRAIN_SRC} folds 1-8","val_same":f"{TRAIN_SRC} fold 9","cross_test":f"{CROSS_SRC} folds 1-8",
 "date_frozen":datetime.now().strftime("%Y-%m-%d %H:%M"),
 "K":int(K),"xgb_params":{**CFG,"scale_pos_weight":float(spw8)},"features":FEATURES_K,
 "selection":"single-dataset gate |d|>0.3 & p_FDR<0.05 (BH) & IC95 excl 0; Spearman>0.9 dedup; K & config by 2D grid + 1-SE parsimony rule (never selects on fold 9)",
 "metrics_same":SAME_METRICS,"metrics_cross":CROSS_METRICS,
 "batch_effect":f"AP same {SAME_METRICS['AP']:.3f} -> cross {CROSS_METRICS['AP']:.3f} (collapse); AUC same {SAME_METRICS['AUC']:.3f} -> cross {CROSS_METRICS['AUC']:.3f} (holds)",
 "legacy_08bc":{"same_AP":LEGACY['same_AP'],"cross_AP":LEGACY['cross_AP'],"note":"re-derived EN + 1-SE selection; supersedes 08b/08c"},
 "notes":"Batch-effect diagnostic detector. F1-max threshold on OOF same. Fold 10 (both datasets) + fold 9 cross NEVER touched.",
}
with open(os.path.join(MDIR,f'm2_{TAG}_config.json'),'w',encoding='utf-8') as f:
    json.dump(config,f,ensure_ascii=False,indent=2)
print(f"=== M2 {TAG} FROZEN -> {MDIR} ===")
print(f"  same AP {SAME_METRICS['AP']:.3f} (legacy {LEGACY['same_AP']}) | cross AP {CROSS_METRICS['AP']:.3f} (legacy {LEGACY['cross_AP']}) | K={K}")
print("  Fold 10 + fold 9 cross never touched.")
